# Multi-Agent Systems with LangGraph

> **Track:** AI Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
Single-agent systems hit limits on complex tasks. Multi-agent systems divide work among specialized agents: a researcher, a writer, a critic — each focused on one thing, collaborating through shared state.

### What You'll Learn
- Why multi-agent vs single-agent
- LangGraph StateGraph concepts
- Defining agent nodes and routing logic
- Researcher, Writer, and Critic agents
- Human-in-the-loop checkpoint
- Compiling and running a workflow

```bash
pip install langgraph langchain langchain-openai
```

In [ ]:
# Setup: define shared state and basic agents
import os
from typing import TypedDict, Annotated
from operator import add
import json

# Simulated agent responses (replace with real LLM calls using langchain_openai)
def mock_llm_call(system: str, user: str) -> str:
    """Stub: returns template response for demo without API calls."""
    if 'research' in system.lower():
        return f'[RESEARCH] Key findings about "{user[:30]}...": 1. Main point A. 2. Main point B. 3. Recent development C.'
    elif 'write' in system.lower():
        return f'[DRAFT] Executive Summary:\n\nBased on research: Main point A drives the narrative...'
    elif 'critic' in system.lower():
        return '[CRITIQUE] The draft is clear but needs: 1. More specific data. 2. A stronger conclusion.'
    return '[AGENT] Task complete.'

print('Multi-agent system setup complete.')
print('Using mock LLM calls — swap mock_llm_call() with real OpenAI/Anthropic calls.')

## 1. Shared State and Graph Definition

In [ ]:
# Define shared state schema for the multi-agent workflow
try:
    from langgraph.graph import StateGraph, END
    from langchain_core.messages import HumanMessage, AIMessage
    LANGGRAPH_AVAILABLE = True
except ImportError:
    LANGGRAPH_AVAILABLE = False
    print('LangGraph not installed. Showing architecture without running.')

class ResearchWorkflowState(TypedDict):
    """Shared state passed between all agents in the workflow."""
    topic: str
    research_notes: str
    draft: str
    critique: str
    revision_count: int
    final_report: str
    status: str  # 'researching', 'writing', 'critiquing', 'done'

print('State schema:')
for field, type_hint in ResearchWorkflowState.__annotations__.items():
    print(f'  {field}: {type_hint.__name__ if hasattr(type_hint, "__name__") else str(type_hint)}')

## 2. Agent Node Implementations

In [ ]:
# Implement the three specialized agent nodes

def researcher_agent(state: ResearchWorkflowState) -> dict:
    """Gathers information about the topic."""
    print(f'\n🔍 Researcher: investigating "{state["topic"]}"')
    notes = mock_llm_call(
        system='You are a thorough researcher. Find key facts, statistics, and recent developments.',
        user=state['topic']
    )
    return {'research_notes': notes, 'status': 'researching'}

def writer_agent(state: ResearchWorkflowState) -> dict:
    """Writes a draft based on research notes."""
    print('\n✍️  Writer: drafting report from research notes')
    draft = mock_llm_call(
        system='You are an expert technical writer. Write a clear, structured report.',
        user=f'Topic: {state["topic"]}\nResearch: {state["research_notes"]}'
    )
    return {'draft': draft, 'status': 'writing'}

def critic_agent(state: ResearchWorkflowState) -> dict:
    """Reviews and critiques the draft."""
    print('\n🔎 Critic: reviewing draft quality')
    critique = mock_llm_call(
        system='You are a critical editor. Identify weaknesses and suggest specific improvements.',
        user=state['draft']
    )
    return {'critique': critique, 'revision_count': state.get('revision_count', 0) + 1, 'status': 'critiquing'}

def finalizer_agent(state: ResearchWorkflowState) -> dict:
    """Produces the final polished report."""
    print('\n✅ Finalizer: producing final report')
    final = f'# Report: {state["topic"]}\n\n{state["draft"]}\n\n---\n*Reviewed {state["revision_count"]} time(s)*'
    return {'final_report': final, 'status': 'done'}

def should_revise(state: ResearchWorkflowState) -> str:
    """Routing function: revise again or finalize?"""
    if state.get('revision_count', 0) < 2 and 'needs' in state.get('critique', '').lower():
        print('\n↩️  Router: sending back for revision')
        return 'revise'
    print('\n→ Router: quality sufficient, finalizing')
    return 'finalize'

print('All agent nodes defined: researcher → writer → critic → [revise|finalize]')

## 3. Building the LangGraph Workflow

In [ ]:
# Assemble the StateGraph
if LANGGRAPH_AVAILABLE:
    graph = StateGraph(ResearchWorkflowState)

    # Add nodes
    graph.add_node('researcher', researcher_agent)
    graph.add_node('writer', writer_agent)
    graph.add_node('critic', critic_agent)
    graph.add_node('finalizer', finalizer_agent)

    # Add edges
    graph.set_entry_point('researcher')
    graph.add_edge('researcher', 'writer')
    graph.add_edge('writer', 'critic')
    graph.add_conditional_edges(
        'critic',
        should_revise,
        {'revise': 'writer', 'finalize': 'finalizer'}
    )
    graph.add_edge('finalizer', END)

    app = graph.compile()
    print('LangGraph workflow compiled successfully.')
    print('Nodes:', list(graph.nodes.keys()))
else:
    print('Simulating workflow execution (LangGraph not installed):')
    # Manual simulation
    state = {'topic': 'The impact of LLMs on software development', 'research_notes': '', 'draft': '', 'critique': '', 'revision_count': 0, 'final_report': '', 'status': ''}
    state.update(researcher_agent(state))
    state.update(writer_agent(state))
    state.update(critic_agent(state))
    route = should_revise(state)
    if route == 'revise':
        state.update(writer_agent(state))
    state.update(finalizer_agent(state))
    print('\nFinal report preview:')
    print(state['final_report'][:300])

## 4. Running the Workflow

In [ ]:
# Execute the multi-agent workflow
topic = 'The impact of LLMs on software development in 2024-2025'

if LANGGRAPH_AVAILABLE:
    initial_state = ResearchWorkflowState(
        topic=topic, research_notes='', draft='', critique='',
        revision_count=0, final_report='', status=''
    )
    final_state = app.invoke(initial_state)
    print('\n=== FINAL REPORT ===')
    print(final_state['final_report'])
    print(f'\nTotal revisions: {final_state["revision_count"]}')
else:
    # Already ran above in simulation
    print('Workflow completed via simulation.')
    print(f'Topic: {topic}')
    print(f'Revisions: {state["revision_count"]}')
    print(f'Status: {state["status"]}')

## Exercises

1. **Add a fact-checker agent**: Insert a `fact_checker` node between the writer and critic. It should identify claims in the draft and verify them against the research notes. Return a list of verified and unverified claims.

2. **Human-in-the-loop**: Use LangGraph's `interrupt_before` parameter to pause the workflow before the finalizer and print the draft to the user. Only proceed when the user types 'approve'. If they type feedback, route back to the writer.

3. **Parallel research**: Modify the graph to have 3 parallel researcher agents (one for 'background', one for 'recent news', one for 'expert opinions'). Use `fan_out` and `fan_in` patterns to merge their outputs before the writer node.